In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [2]:
class Net(nn.Module):
    """Model (simple CNN adapted from 'PyTorch: A 60 Minute Blitz')"""

    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


In [4]:
model = Net()

In [13]:
print(model.parameters)

<bound method Module.parameters of Net(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)>


In [17]:
for p in model.parameters():
    print(p.shape)
    print("moving on to next p")

torch.Size([6, 3, 5, 5])
moving on to next p
torch.Size([6])
moving on to next p
torch.Size([16, 6, 5, 5])
moving on to next p
torch.Size([16])
moving on to next p
torch.Size([120, 400])
moving on to next p
torch.Size([120])
moving on to next p
torch.Size([84, 120])
moving on to next p
torch.Size([84])
moving on to next p
torch.Size([10, 84])
moving on to next p
torch.Size([10])
moving on to next p


In [8]:
serialized_p  = [torch.zeros_like(p).tolist() for p in model.parameters()]

In [33]:
print(len(serialized_p))
for i in range(len(serialized_p)):
  print(len(serialized_p[i]))

10
6
6
16
16
120
120
84
84
10
10


In [36]:
print(len(serialized_p[0]))
print(len(serialized_p[0][0]))
print(len(serialized_p[0][0][0]))
print(len(serialized_p[0][0][0][0]))

6
3
5
5


In [38]:
#serialization seems to be correct

In [40]:
# test deserialization
c_i  = [torch.tensor(layer) for layer in serialized_p]

In [46]:
print(c_i[2].shape)

torch.Size([16, 6, 5, 5])


In [ ]:
#deserialization seems to be correct as well with only diff. beig that keep in mind that this is a list of the tensors
#not an iterable over them as in model.parameters

In [59]:
server_control_variate = { "control-variate": [torch.zeros_like(p).tolist() for p in model.parameters()] }


In [60]:
c  = [torch.tensor(layer) for layer in server_control_variate["control-variate"]]

In [61]:
print(c)

[tensor([[[[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]]],


        [[[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]],

         [[0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0.]]],


        [[[0., 0., 0., 

In [62]:
print(len(c))
for i in range(len(c)):
  print(len(c[i]))

10
6
6
16
16
120
120
84
84
10
10


In [63]:
print(len(c[0]))
print(len(c[0][0]))
print(len(c[0][0][0]))
print(len(c[0][0][0][0]))

6
3
5
5


In [64]:
config = {}

In [65]:
#encode sendable format
config["server-control-variate"] = {
    "shapes": [len(layer) for layer in server_control_variate["control-variate"]],
    "values": [
        v
        for layer in server_control_variate["control-variate"]
        for v in layer
    ],
}

In [66]:
#decoding at client
vals = config["server-control-variate"]["values"]
shapes = config["server-control-variate"]["shapes"]

c = []
idx = 0
for s in shapes:
    c.append(torch.tensor(vals[idx:idx+s]))
    idx += s

In [67]:
print(len(c))
for i in range(len(c)):
  print(len(c[i]))

10
6
6
16
16
120
120
84
84
10
10


In [69]:
print(len(c[0]))
print(len(c[0][0]))
print(len(c[0][0][0]))
print(len(c[0][0][0][0]))

6
3
5
5
